# E-commerce GMV Growth Attribution Analysis

Objectives:
1. Load multiple CSVs: user_behavior / order / order_item / product / user / platform
2. Build core e-commerce metrics: GMV, order count, buyers, AOV, impressions, clicks, add-to-cart, CTR, CVR
3. Attribution analysis across platform, city, category, and product dimensions
4. Decompose GMV via multiplicative model: GMV = Exposure × CTR × CVR × AOV
5. Export structured JSON output
6. Generate Feishu card HTML for business result delivery

Notes:
- No `country` field available; using `city` as the geographic dimension proxy.
- No explicit `impression` field; using `view/browse` behavior as impression proxy.


---
## 0. Environment Setup


In [1]:
# Install dependencies (first run only)
# !pip install pandas numpy matplotlib seaborn plotly shap requests ipython -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json
import requests
import warnings
from IPython.display import display, HTML, JSON
from itertools import combinations
from collections import defaultdict

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('✅ Dependencies loaded')


✅ Dependencies loaded


---
## 1. Data Loading & Exploration


In [2]:

DATA_DIR = './ecommerce dataset/'

# Load all tables
platform_df      = pd.read_csv(DATA_DIR + 'platform.csv')
user_df          = pd.read_csv(DATA_DIR + 'user.csv')
product_df       = pd.read_csv(DATA_DIR + 'product.csv')
behavior_df      = pd.read_csv(DATA_DIR + 'user_behavior.csv', parse_dates=['behavior_time'])
order_df         = pd.read_csv(DATA_DIR + 'order.csv', parse_dates=['order_time', 'payment_time'])
order_item_df    = pd.read_csv(DATA_DIR + 'order_item.csv')

print('=== Row counts per table ===')
for name, df in [
    ('platform', platform_df), ('user', user_df),
    ('product', product_df), ('user_behavior', behavior_df),
    ('order', order_df), ('order_item', order_item_df)
]:
    print(f'  {name:20s}  {len(df):>8,}  rows  {df.shape[1]} cols')


=== Row counts per table ===
  platform                     5  rows  5 cols
  user                     5,486  rows  11 cols
  product                  3,298  rows  10 cols
  user_behavior           21,998  rows  15 cols
  order                   10,996  rows  15 cols
  order_item              27,523  rows  7 cols


In [3]:
# Quick data quality check
print('=== User Behavior Table - Behavior Type Distribution ===')
display(behavior_df['behavior_type'].value_counts().to_frame('count'))

print('\n=== Order Table - Status Distribution ===')
display(order_df['order_status'].value_counts().to_frame('count'))

print('\n=== Product Table - Category Distribution ===')
display(product_df['category'].value_counts().to_frame('count'))


=== User Behavior Table - Behavior Type Distribution ===


,count
behavior_type,
浏览,5458
加购,4034
收藏,3276
下单,2810
支付,2428
评价,2102
分享,1890



=== Order Table - Status Distribution ===


,count
order_status,
待付款,2224
已完成,2216
已付款,2192
已发货,2187
已取消,2177



=== Product Table - Category Distribution ===


,count
category,
母婴用品,446
图书文具,426
运动户外,418
食品饮料,416
美妆个护,413
电子产品,397
家居用品,391
服装鞋帽,391


In [4]:
# Date range check
print(f'Behavior data date range：{behavior_df["behavior_time"].min()} → {behavior_df["behavior_time"].max()}')
print(f'Order data date range：{order_df["order_time"].min()} → {order_df["order_time"].max()}')

# Slice by month for MoM/YoY comparison later
behavior_df['ym'] = behavior_df['behavior_time'].dt.to_period('M')
order_df['ym']    = order_df['order_time'].dt.to_period('M')


Behavior data date range：2025-01-01 01:43:41 → 2026-12-28 23:28:31
Order data date range：2025-01-01 01:06:28 → 2026-12-28 22:33:43


---
## 2. Build Funnel Metrics Table

Aggregate behavior stages (Impression → Click → Add-to-Cart → Order) into a wide-format funnel table for multiplicative decomposition.


In [5]:
# ── Behavior type normalization ────────────────────────────────
# No 'click' behavior in data; using 'add-to-cart' as CTR numerator (add-to-cart rate)
# CTR = cart/view,  CVR = order/cart,  payment_rate = payment/order
IMPRESSION_TYPES = ['浏览', 'view', 'impression', 'pv']
CART_TYPES       = ['加购', 'cart', 'add_to_cart']
ORDER_TYPES      = ['下单', 'buy', 'order', 'purchase']
PAYMENT_TYPES    = ['支付', 'pay', 'payment']
FAVORITE_TYPES   = ['收藏', 'favorite', 'wishlist', 'collect']
REVIEW_TYPES     = ['评价', 'review', 'comment', 'rate']
SHARE_TYPES      = ['分享', 'share']
CLICK_TYPES      = ['点击', 'click']   # Keep mapping — auto-activates if click behavior appears in future data

def classify_behavior(bt):
    bt_lower = str(bt).lower()
    if any(t in bt_lower for t in IMPRESSION_TYPES): return 'impression'
    if any(t in bt_lower for t in CLICK_TYPES):      return 'click'
    if any(t in bt_lower for t in CART_TYPES):       return 'cart'
    if any(t in bt_lower for t in ORDER_TYPES):      return 'order_behavior'
    if any(t in bt_lower for t in PAYMENT_TYPES):    return 'payment'
    if any(t in bt_lower for t in FAVORITE_TYPES):   return 'favorite'
    if any(t in bt_lower for t in REVIEW_TYPES):     return 'review'
    if any(t in bt_lower for t in SHARE_TYPES):      return 'share'
    return 'other'

behavior_df['stage'] = behavior_df['behavior_type'].apply(classify_behavior)
print('Behavior stage distribution:')
display(behavior_df['stage'].value_counts())


Behavior stage distribution:


stage
impression        5458
cart              4034
favorite          3276
order_behavior    2810
payment           2428
review            2102
share             1890
Name: count, dtype: int64

In [6]:
# ── JOIN: order + order_item + product ──────────────────────────
order_full = (
    order_df
    .merge(order_item_df, on='order_id', how='left')
    .merge(product_df[['global_product_id', 'category', 'subcategory', 'brand', 'price']],
           on='global_product_id', how='left')
    .merge(user_df[['global_user_id', 'city', 'user_level']],
           on='global_user_id', how='left')
)

# Keep only valid orders (exclude cancelled / refunded)
VALID_STATUS = ['已完成', '已支付', '待发货', '已发货', 'completed', 'paid', 'shipped']
order_valid = order_full[order_full['order_status'].isin(VALID_STATUS)].copy()
print(f'Valid order rows: {len(order_valid):,}')
print(f'Valid order total GMV: ¥{order_valid["item_total"].sum():,.2f}')


Valid order rows: 11,046
Valid order total GMV: ¥11,188,939.44


In [7]:
# ── Funnel wide table (by platform + category + month) ──────────
def build_funnel_table(dim_cols):
    """Build funnel metrics wide table for any dimension combination"""
    beh = behavior_df.copy()
    beh = beh.merge(product_df[['global_product_id', 'category']],
                    on='global_product_id', how='left')
    beh = beh.merge(user_df[['global_user_id', 'city']], on='global_user_id', how='left')

    stage_pivot = (
        beh.groupby(dim_cols + ['stage'])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )
    # Ensure all stage columns exist
    for col in ['impression', 'click', 'cart', 'order_behavior',
                'payment', 'favorite', 'review', 'share']:
        if col not in stage_pivot.columns:
            stage_pivot[col] = 0

    gmv = (
        order_valid
        .groupby(dim_cols)
        .agg(
            gmv=('item_total', 'sum'),
            order_count=('order_id', 'nunique'),
            item_count=('order_item_id', 'count')
        )
        .reset_index()
    )

    funnel = stage_pivot.merge(gmv, on=dim_cols, how='outer').fillna(0)

    # CTR = cart/view (no click data; using add-to-cart rate as proxy)
    funnel['ctr']          = np.where(funnel['impression'] > 0,
                                       funnel['cart'] / funnel['impression'], 0)
    # CVR = order/cart
    funnel['cvr']          = np.where(funnel['cart'] > 0,
                                       funnel['order_behavior'] / funnel['cart'], 0)
    # new metrics
    funnel['payment_rate'] = np.where(funnel['order_behavior'] > 0,
                                       funnel['payment'] / funnel['order_behavior'], 0)
    funnel['favorite_rate']= np.where(funnel['impression'] > 0,
                                       funnel['favorite'] / funnel['impression'], 0)
    funnel['review_rate']  = np.where(funnel['payment'] > 0,
                                       funnel['review'] / funnel['payment'], 0)
    funnel['share_rate']   = np.where(funnel['impression'] > 0,
                                       funnel['share'] / funnel['impression'], 0)
    funnel['arpu']         = np.where(funnel['order_count'] > 0,
                                       funnel['gmv'] / funnel['order_count'], 0)
    funnel['ym'] = funnel['ym'].astype(str)
    return funnel

funnel_platform  = build_funnel_table(['platform', 'ym'])
funnel_category  = build_funnel_table(['category', 'ym'])
funnel_city      = build_funnel_table(['city', 'ym'])

print('✅ Funnel metrics table built')
print('Note: CTR = cart/view (add-to-cart rate), CVR = order/cart (conversion rate)')
funnel_platform.head(3)



✅ Funnel metrics table built
Note: CTR = cart/view (add-to-cart rate), CVR = order/cart (conversion rate)


,platform,ym,cart,favorite,impression,order_behavior,payment,review,share,click,gmv,order_count,item_count,ctr,cvr,payment_rate,favorite_rate,review_rate,share_rate,arpu
0,douyin,2025-01,62,59,82,42,31,25,18,0,161641.73,59,144,0.756098,0.677419,0.738095,0.719512,0.806452,0.219512,2739.690339
1,douyin,2025-02,42,42,72,28,24,21,19,0,135785.53,53,128,0.583333,0.666667,0.857143,0.583333,0.875000,0.263889,2561.991132
2,douyin,2025-03,65,48,72,36,42,27,22,0,141533.06,58,146,0.902778,0.553846,1.166667,0.666667,0.642857,0.305556,2440.225172


---
## 3. GMV Multiplicative Decomposition & Attribution

$$GMV = \underbrace{Impression}_{\text{Traffic}} \times \underbrace{CTR}_{\text{CTR}} \times \underbrace{CVR}_{\text{CVR}} \times \underbrace{ARPU}_{\text{ARPU}}$$

GMV MoM Δ = sum of factor contributions (multiplicative decomposition → log-difference)


In [8]:
def gmv_decompose(df, dim_col):
    """
    Single-dimension GMV multiplicative attribution:
    GMV = impression × CTR × CVR × ARPU
    Log-difference method to quantify each factor's MoM contribution
    """
    df = df.copy()
    df['ym_str'] = df['ym'].astype(str)
    df = df.sort_values([dim_col, 'ym_str'])

    # Log-transform (clip near-zero values)
    for col in ['impression', 'ctr', 'cvr', 'arpu']:
        df[f'log_{col}'] = np.log(df[col].clip(lower=1e-9))

    # MoM diff = difference of log values per factor
    df['log_gmv'] = np.log(df['gmv'].clip(lower=1e-9))
    for col in ['impression', 'ctr', 'cvr', 'arpu', 'gmv']:
        df[f'd_{col}'] = df.groupby(dim_col)[f'log_{col}' if col != 'gmv' else 'log_gmv'].diff()

    result = df.dropna(subset=['d_gmv']).copy()

    # Contribution percentage per factor
    factors = ['impression', 'ctr', 'cvr', 'arpu']
    for f in factors:
        result[f'contrib_{f}'] = np.where(
            result['d_gmv'].abs() > 1e-9,
            result[f'd_{f}'] / result['d_gmv'].abs(),
            0
        )

    return result[[dim_col, 'ym_str', 'gmv', 'd_gmv',
                   'impression', 'ctr', 'cvr', 'arpu'] +
                  [f'contrib_{f}' for f in factors]]


decomp_platform = gmv_decompose(funnel_platform, 'platform')
decomp_category = gmv_decompose(funnel_category, 'category')
decomp_city     = gmv_decompose(funnel_city,     'city')

print('GMV multiplicative decomposition complete. Sample:')
display(decomp_platform.head())


GMV multiplicative decomposition complete. Sample:


,platform,ym_str,gmv,d_gmv,impression,ctr,cvr,arpu,contrib_impression,contrib_ctr,contrib_cvr,contrib_arpu
1,douyin,2025-02,135785.53,-0.174306,72,0.583333,0.666667,2561.991132,-0.746121,-1.488257,-0.091795,-0.384727
2,douyin,2025-03,141533.06,0.041457,72,0.902778,0.553846,2440.225172,0.000000,10.534315,-4.472217,-1.174586
3,douyin,2025-04,120286.61,-0.162656,88,0.795455,0.800000,2405.732200,1.233712,-0.778100,2.260751,-0.087522
4,douyin,2025-05,180590.88,0.406357,75,0.693333,0.692308,2655.748235,-0.393370,-0.338133,-0.355799,0.243314
5,douyin,2025-06,105119.47,-0.541137,80,0.637500,0.803922,2627.986750,0.119265,-0.155149,0.276217,-0.019419


In [9]:
# ── Visualization: Monthly GMV trend by platform ────────────────
plot_df = funnel_platform.copy()
plot_df['ym'] = plot_df['ym'].astype(str)
plot_df = plot_df.sort_values('ym')

fig = px.line(
    plot_df,
    x='ym', y='gmv', color='platform',
    title='Monthly GMV Trend by Platform',
    labels={'ym': 'Month', 'gmv': 'GMV (CNY)', 'platform': 'Platform'}
)
fig.update_layout(template='plotly_white', height=400)
fig.show()


In [10]:
# ── Visualization: Stacked attribution bar chart (platform) ─────
latest = decomp_platform.copy()
factors = ['contrib_impression', 'contrib_ctr', 'contrib_cvr', 'contrib_arpu']
factor_labels = {'contrib_impression': 'Impression', 'contrib_ctr': 'CTR',
                  'contrib_cvr': 'CVR', 'contrib_arpu': 'ARPU'}

fig2 = go.Figure()
colors = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2']
for f, c in zip(factors, colors):
    fig2.add_trace(go.Bar(
        name=factor_labels[f],
        x=latest['ym_str'],
        y=latest[f],
        marker_color=c
    ))

fig2.update_layout(
    barmode='relative',
    title='GMV MoM Attribution Decomposition (Multiplicative)',
    xaxis_title='Month',
    yaxis_title='Contribution (relative)',
    template='plotly_white',
    height=450,
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig2.show()


---
## 4. Multi-Dimensional Drill-Down Analysis

Cross-dimensional attribution across platform × category to identify high- and low-growth segments


In [11]:
# Build platform × category 2D funnel table
funnel_2d = build_funnel_table(['platform', 'category', 'ym'])

summary_2d = (
    funnel_2d.groupby(['platform', 'category'])
    .agg(
        total_gmv=('gmv', 'sum'),
        total_impression=('impression', 'sum'),
        total_cart=('cart', 'sum'),
        total_order=('order_behavior', 'sum'),
        total_payment=('payment', 'sum'),
        total_favorite=('favorite', 'sum'),
    )
    .reset_index()
)
summary_2d['ctr']          = summary_2d['total_cart']    / summary_2d['total_impression'].clip(lower=1)
summary_2d['cvr']          = summary_2d['total_order']   / summary_2d['total_cart'].clip(lower=1)
summary_2d['payment_rate'] = summary_2d['total_payment'] / summary_2d['total_order'].clip(lower=1)
summary_2d['favorite_rate']= summary_2d['total_favorite']/ summary_2d['total_impression'].clip(lower=1)

pivot_gmv = summary_2d.pivot_table(index='category', columns='platform',
                                    values='total_gmv', fill_value=0)
fig3 = px.imshow(
    pivot_gmv,
    title='Platform × Category GMV Heatmap',
    color_continuous_scale='Blues',
    aspect='auto'
)
fig3.update_layout(height=500)
fig3.show()



In [12]:
# ── Auxiliary Behavior Metric Changes（wishlist_rate / share_rate / review_rate）────────────────────
# Take all platforms — latest two months, compute MoM change
periods = sorted(funnel_platform['ym'].unique())
prev_ym, curr_ym = periods[-2], periods[-1]

def get_aux_rates(ym_val):
    sub = funnel_platform[funnel_platform['ym'] == ym_val]
    imp = sub['impression'].sum()
    pay = sub['payment'].sum()
    return {
        'wishlist_rate': sub['favorite'].sum() / imp if imp > 0 else 0,
        'share_rate': sub['share'].sum()    / imp if imp > 0 else 0,
        'review_rate': sub['review'].sum()   / pay if pay > 0 else 0,
    }

prev_rates = get_aux_rates(prev_ym)
curr_rates = get_aux_rates(curr_ym)

rows_html = ''
for name in ['wishlist_rate', 'share_rate', 'review_rate']:
    prev_v  = prev_rates[name]
    curr_v  = curr_rates[name]
    abs_chg = curr_v - prev_v
    rel_chg = abs_chg / prev_v if prev_v > 0 else 0
    color   = '#16a34a' if abs_chg >= 0 else '#dc2626'
    abs_s   = '+' if abs_chg >= 0 else ''
    rel_s   = '+' if rel_chg >= 0 else ''
    rows_html += f"""
    <tr style="border-bottom:1px solid #f1f5f9">
      <td style="padding:14px 16px;font-weight:500">{name}</td>
      <td style="padding:14px 16px;color:#64748b">{prev_v:.2%}</td>
      <td style="padding:14px 16px;color:#64748b">{curr_v:.2%}</td>
      <td style="padding:14px 16px;color:{color};font-weight:700">{abs_s}{abs_chg:.2%}</td>
      <td style="padding:14px 16px;color:{color};font-weight:700">{rel_s}{rel_chg:.2%}</td>
    </tr>"""

display(HTML(f"""
<div style="background:#fff;border-radius:14px;padding:28px;margin:16px 0;
            box-shadow:0 2px 10px rgba(0,0,0,.05)">
  <h2 style="font-size:18px;font-weight:700;color:#1e293b;margin-bottom:8px">
    Auxiliary Behavior Metric Changes
  </h2>
  <p style="font-size:13px;color:#64748b;margin-bottom:20px">
    收藏、分享、review excluded from GMV main funnel factors in multiplicative decomposition; treated as user engagement metrics to explain interest & post-purchase behavior.
  </p>
  <table style="width:100%;border-collapse:collapse;font-size:14px">
    <thead>
      <tr style="border-bottom:2px solid #e2e8f0;color:#475569;font-weight:600">
        <th style="padding:10px 16px;text-align:left">Metric</th>
        <th style="padding:10px 16px;text-align:left">Previous ({prev_ym})</th>
        <th style="padding:10px 16px;text-align:left">Current ({curr_ym})</th>
        <th style="padding:10px 16px;text-align:left">MoM Change (abs)</th>
        <th style="padding:10px 16px;text-align:left">MoM Change (%)</th>
      </tr>
    </thead>
    <tbody>{rows_html}</tbody>
  </table>
</div>
"""))



Metric,Previous (2026-11),Current (2026-12),MoM Change (abs),MoM Change (%)
wishlist_rate,60.92%,63.80%,+2.88%,+4.72%
share_rate,34.45%,31.22%,-3.23%,-9.38%
review_rate,82.83%,93.68%,+10.86%,+13.11%


In [13]:
# Scatter: CTR vs CVR, bubble size = GMV, color = platform
fig4 = px.scatter(
    summary_2d,
    x='ctr', y='cvr',
    size='total_gmv', color='platform',
    hover_data=['category', 'total_gmv'],
    title='CTR vs CVR by Dimension (bubble size = GMV)',
    labels={'ctr': 'Add-to-Cart Rate (CTR)', 'cvr': 'Conversion Rate (CVR)'}
)
fig4.update_layout(template='plotly_white', height=450)
fig4.show()


---
## 5. Algorithm: Shapley Value Factor Attribution

Apply Shapley Value (game-theoretic method) to fairly attribute each factor's contribution to GMV growth, resolving interaction effects in multiplicative decomposition


In [14]:
def compute_shapley(row, factors):
    """
    Simplified Shapley Value computation.
    factors: dict of {factor_name: delta_value} (per-factor deltas)
    GMV change ≈ prod(1 + delta_i) - 1, decomposed fairly across factors
    """
    n = len(factors)
    names = list(factors.keys())
    deltas = list(factors.values())
    shapley = defaultdict(float)

    from itertools import combinations
    from math import factorial

    for i, name in enumerate(names):
        for size in range(n):
            # All subsets not containing player i
            others = [j for j in range(n) if j != i]
            subsets = list(combinations(others, size))
            weight = factorial(size) * factorial(n - size - 1) / factorial(n)
            for subset in subsets:
                # v(S∪{i}) - v(S)
                s_vals = [deltas[j] for j in subset]
                si_vals = s_vals + [deltas[i]]
                v_s  = np.prod([1 + d for d in s_vals]) - 1  if s_vals  else 0
                v_si = np.prod([1 + d for d in si_vals]) - 1
                shapley[name] += weight * (v_si - v_s)
    return shapley


# Demo: compute delta from the latest period for one platform
# Get latest two periods for a specific platform to compute delta
def get_latest_deltas(df, dim_col, dim_val):
    sub = df[df[dim_col] == dim_val].sort_values('ym_str').tail(1)
    if sub.empty:
        return None
    row = sub.iloc[0]
    return {
        'Impression': row['contrib_impression'],
        'CTR':  row['contrib_ctr'],
        'CVR':  row['contrib_cvr'],
        'ARPU': row['contrib_arpu'],
    }


# Compute Shapley for each platform
shapley_results = []
for platform in decomp_platform['platform'].unique():
    deltas = get_latest_deltas(decomp_platform, 'platform', platform)
    if deltas is None:
        continue
    sv = compute_shapley(None, deltas)
    sv['platform'] = platform
    shapley_results.append(sv)

shapley_df = pd.DataFrame(shapley_results).set_index('platform')
print('Shapley attribution by platform:')
display(shapley_df.style.background_gradient(cmap='RdYlGn', axis=1).format('{:.3f}'))


Shapley attribution by platform:


,Impression,CTR,CVR,ARPU
platform,,,,
douyin,-1.381,5.673,9.140,5.458
jd,-0.610,0.168,-0.286,-0.212
taobao,0.050,-0.008,-1.208,0.099


In [15]:
# Visualize Shapley contributions
fig5 = px.bar(
    shapley_df.reset_index().melt(id_vars='platform'),
    x='platform', y='value', color='variable',
    barmode='group',
    title='Shapley Factor Attribution by Platform',
    labels={'value': 'Contribution', 'variable': 'Factor', 'platform': 'Platform'}
)
fig5.update_layout(template='plotly_white', height=400)
fig5.show()


---
## 7. Feishu Card JSON Generation


In [16]:
def build_feishu_card(summary_data, model_responses):
    """
    Generate Feishu card JSON (compatible with Feishu Message Card v2 format)
    Docs: https://open.feishu.cn/document/uAjLw4CM/ukzMukzMukzM/feishu-cards/card-components
    """

    # Extract key metrics
    top_platform = funnel_platform.groupby('platform')['gmv'].sum().idxmax()
    top_category = funnel_category.groupby('category')['gmv'].sum().idxmax()
    total_gmv    = order_valid['item_total'].sum()
    avg_ctr      = funnel_platform['ctr'].mean()
    avg_cvr      = funnel_platform['cvr'].mean()

    card = {
        "schema": "2.0",
        "config": {
            "wide_screen_mode": True,
            "enable_forward": True
        },
        "header": {
            "title": {
                "tag": "plain_text",
                "content": "📊 E-commerce GMV Growth Attribution Report"
            },
            "subtitle": {
                "tag": "plain_text",
                "content": f"Period: Recent | Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}"
            },
            "template": "blue"
        },
        "body": {
            "elements": [
                # ── KPI Cards ──
                {
                    "tag": "column_set",
                    "flex_mode": "none",
                    "background_style": "grey",
                    "columns": [
                        {
                            "tag": "column",
                            "width": "weighted",
                            "weight": 1,
                            "elements": [{
                                "tag": "div",
                                "text": {
                                    "tag": "lark_md",
                                    "content": f"**Total GMV**\n💰 ¥{total_gmv:,.0f}"
                                }
                            }]
                        },
                        {
                            "tag": "column",
                            "width": "weighted",
                            "weight": 1,
                            "elements": [{
                                "tag": "div",
                                "text": {
                                    "tag": "lark_md",
                                    "content": f"**Avg CTR**\n🖱️ {avg_ctr:.2%}"
                                }
                            }]
                        },
                        {
                            "tag": "column",
                            "width": "weighted",
                            "weight": 1,
                            "elements": [{
                                "tag": "div",
                                "text": {
                                    "tag": "lark_md",
                                    "content": f"**Avg CVR**\n🛒 {avg_cvr:.2%}"
                                }
                            }]
                        },
                        {
                            "tag": "column",
                            "width": "weighted",
                            "weight": 1,
                            "elements": [{
                                "tag": "div",
                                "text": {
                                    "tag": "lark_md",
                                    "content": f"**Top Platform**\n🏆 {top_platform}"
                                }
                            }]
                        }
                    ]
                },
                {"tag": "hr"},
                # ── GMV Decomposition Notes ──
                {
                    "tag": "div",
                    "text": {
                        "tag": "lark_md",
                        "content": "**🔍 GMV Multiplicative Decomposition Model**\nGMV = Impression × CTR (Add-to-Cart Rate) × CVR (Conversion Rate) × ARPU (Avg Order Value)\n\n**Attribution Method:** Log-difference decomposition + Shapley Value fair allocation"
                    }
                },
                {"tag": "hr"},
                # ── Drill-Down Results ──
                {
                    "tag": "div",
                    "text": {
                        "tag": "lark_md",
                        "content": f"**📦 Multi-Dimensional Analysis Results**\n- **Highest GMVCategory：** {top_category}\n- **Highest GMVPlatform：** {top_platform}\n- **Dimensions:** Platform / Category / City"
                    }
                },
                {"tag": "hr"},
                
            ]
        }
    }
    return card


feishu_card = build_feishu_card({}, {})
print('✅ Feishu card JSON generated')
display(JSON(feishu_card))


✅ Feishu card JSON generated


<IPython.core.display.JSON object>

In [17]:
# Save Feishu card JSON
with open('feishu_card.json', 'w', encoding='utf-8') as f:
    json.dump(feishu_card, f, ensure_ascii=False, indent=2)
print('Feishu card saved to feishu_card.json')


Feishu card saved to feishu_card.json


---
## 8. HTML Highlighted Report Generation


In [18]:
def generate_html_report():
    top_platform = funnel_platform.groupby('platform')['gmv'].sum().idxmax()
    top_category = funnel_category.groupby('category')['gmv'].sum().idxmax()
    total_gmv    = order_valid['item_total'].sum()
    avg_ctr      = funnel_platform['ctr'].mean()
    avg_cvr      = funnel_platform['cvr'].mean()

    plat_rank = funnel_platform.groupby('platform')['gmv'].sum().sort_values(ascending=False)
    max_gmv = plat_rank.max()

    platform_rows = ''
    for i, (plat, gmv) in enumerate(plat_rank.items()):
        bar_pct = gmv / max_gmv * 100
        highlight = 'highlight-top' if i == 0 else ''
        platform_rows += f"""
        <tr class="{highlight}">
            <td>{'🥇' if i==0 else '🥈' if i==1 else '🥉' if i==2 else str(i+1)} {plat}</td>
            <td>¥{gmv:,.0f}</td>
            <td><div class="bar-container"><div class="bar" style="width:{bar_pct:.1f}%"></div></div></td>
        </tr>"""

    cat_rank = funnel_category.groupby('category')['gmv'].sum().sort_values(ascending=False).head(10)
    category_rows = ''
    max_cat_gmv = cat_rank.max()
    for i, (cat, gmv) in enumerate(cat_rank.items()):
        bar_pct = gmv / max_cat_gmv * 100
        highlight = 'highlight-top' if i == 0 else ''
        avg_ctr_cat = funnel_category[funnel_category['category']==cat]['ctr'].mean()
        avg_cvr_cat = funnel_category[funnel_category['category']==cat]['cvr'].mean()
        category_rows += f"""
        <tr class="{highlight}">
            <td>{i+1}. {cat}</td>
            <td>¥{gmv:,.0f}</td>
            <td>{avg_ctr_cat:.2%}</td>
            <td>{avg_cvr_cat:.2%}</td>
            <td><div class="bar-container"><div class="bar" style="width:{bar_pct:.1f}%"></div></div></td>
        </tr>"""

    shapley_html = ''
    if 'shapley_df' in globals():
        for platform in shapley_df.index:
            row = shapley_df.loc[platform]
            cells = ''
            for factor, val in row.items():
                color = '#2e7d32' if val > 0 else '#c62828'
                cells += f'<td style="color:{color};font-weight:600">{val:+.3f}</td>'
            shapley_html += f'<tr><td><strong>{platform}</strong></td>{cells}</tr>'

    html = f"""<!DOCTYPE html>
<html lang="zh">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width,initial-scale=1.0">
  <title>E-commerce GMV Attribution Analysis Report</title>
  <style>
    @import url('https://fonts.googleapis.com/css2?family=Noto+Sans+SC:wght@400;600;700&display=swap');
    *{{box-sizing:border-box;margin:0;padding:0}}
    body{{font-family:'Noto Sans SC',sans-serif;background:#f0f4f8;color:#1e293b;padding:28px;line-height:1.6}}
    .header{{background:linear-gradient(135deg,#0f172a 0%,#1e3a5f 60%,#1d4ed8 100%);color:#fff;padding:40px 36px;border-radius:20px;margin-bottom:24px;box-shadow:0 12px 40px rgba(29,78,216,.35)}}
    .header h1{{font-size:26px;font-weight:700;letter-spacing:.5px}}
    .header .sub{{opacity:.75;margin-top:6px;font-size:13px}}
    .formula-box{{background:#fff;border-radius:14px;padding:22px 28px;margin-bottom:24px;text-align:center;border:2px solid #dbeafe;box-shadow:0 2px 10px rgba(0,0,0,.05)}}
    .formula{{font-size:18px;color:#1e40af;font-weight:700;letter-spacing:.5px}}
    .formula span{{color:#64748b;font-size:13px;font-weight:400}}
    .formula-sub{{margin-top:8px;font-size:12px;color:#94a3b8}}
    .kpi-grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:16px;margin-bottom:24px}}
    .kpi{{background:#fff;border-radius:14px;padding:22px 20px;box-shadow:0 2px 10px rgba(0,0,0,.05);border-top:4px solid}}
    .kpi.gmv{{border-color:#1d4ed8}}.kpi.ctr{{border-color:#059669}}.kpi.cvr{{border-color:#d97706}}.kpi.plat{{border-color:#7c3aed}}
    .kpi-label{{font-size:11px;color:#94a3b8;text-transform:uppercase;letter-spacing:1px;font-weight:600}}
    .kpi-value{{font-size:30px;font-weight:700;margin-top:6px;color:#0f172a}}
    .kpi-sub{{font-size:12px;color:#94a3b8;margin-top:2px}}
    .section{{background:#fff;border-radius:14px;padding:26px;margin-bottom:24px;box-shadow:0 2px 10px rgba(0,0,0,.05)}}
    .section h2{{font-size:17px;font-weight:700;color:#1e293b;margin-bottom:18px;display:flex;align-items:center;gap:8px}}
    table{{width:100%;border-collapse:collapse;font-size:13.5px}}
    th{{background:#f1f5f9;color:#475569;padding:10px 14px;text-align:left;font-weight:600;font-size:12px;text-transform:uppercase;letter-spacing:.5px}}
    td{{padding:11px 14px;border-bottom:1px solid #f1f5f9;vertical-align:middle}}
    tr:last-child td{{border-bottom:none}}
    tr:hover td{{background:#f8fafc}}
    tr.highlight-top td{{background:#fef3c7;font-weight:700}}
    .bar-container{{background:#e2e8f0;border-radius:6px;height:8px;width:160px}}
    .bar{{background:linear-gradient(90deg,#1d4ed8,#60a5fa);height:100%;border-radius:6px}}
    .tag{{display:inline-block;background:#eff6ff;color:#1d4ed8;border-radius:6px;padding:3px 10px;font-size:12px;margin:3px;font-weight:600;border:1px solid #bfdbfe}}
    .funnel-row{{display:flex;align-items:center;gap:0;margin:16px 0}}
    .funnel-step{{flex:1;text-align:center;padding:16px 8px;position:relative}}
    .funnel-step:not(:last-child)::after{{content:'→';position:absolute;right:-10px;top:50%;transform:translateY(-50%);font-size:20px;color:#94a3b8}}
    .funnel-num{{font-size:22px;font-weight:700;color:#1d4ed8}}
    .funnel-label{{font-size:11px;color:#64748b;margin-top:4px;text-transform:uppercase;letter-spacing:.5px}}
    .funnel-rate{{font-size:12px;color:#059669;font-weight:600;margin-top:2px}}
    .shapley-pos{{color:#059669;font-weight:700}}
    .shapley-neg{{color:#dc2626;font-weight:700}}
    .footer{{text-align:center;color:#94a3b8;font-size:12px;margin-top:16px}}
  </style>
</head>
<body>

<div class="header">
  <h1>E-commerce GMV Growth Attribution Report</h1>
  <p class="sub">Data: Multi-platform e-commerce dataset | Generated: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')} | Framework: Multiplicative Decomposition + Shapley Attribution</p>
</div>

<div class="formula-box">
  <div class="formula">GMV &nbsp;=&nbsp; Impression &nbsp;×&nbsp; CTR <span>(Add-to-Cart Rate)</span> &nbsp;×&nbsp; CVR <span>(Conversion Rate)</span> &nbsp;×&nbsp; ARPU <span>(Avg Order Value)</span></div>
  <div class="formula-sub">Method: Log-difference decomposition → MoM factor contribution | Shapley Value → fair multi-factor attribution</div>
</div>

<div class="kpi-grid">
  <div class="kpi gmv"><div class="kpi-label">Total GMV</div><div class="kpi-value">¥{total_gmv:,.0f}</div><div class="kpi-sub">Valid orders total</div></div>
  <div class="kpi ctr"><div class="kpi-label">Avg CTR</div><div class="kpi-value">{avg_ctr:.1%}</div><div class="kpi-sub">Add-to-cart rate avg</div></div>
  <div class="kpi cvr"><div class="kpi-label">Avg CVR</div><div class="kpi-value">{avg_cvr:.1%}</div><div class="kpi-sub">Conversion rate avg</div></div>
  <div class="kpi plat"><div class="kpi-label">Top Platform</div><div class="kpi-value" style="font-size:18px">{top_platform}</div><div class="kpi-sub">Highest GMV</div></div>
</div>

<div class="section">
  <h2>Conversion Funnel Overview</h2>
  <div class="funnel-row">
    <div class="funnel-step" style="background:#eff6ff;border-radius:10px">
      <div class="funnel-num">{int(funnel_platform['impression'].sum()):,}</div>
      <div class="funnel-label">Impression</div>
    </div>
    <div class="funnel-step" style="background:#f0fdf4;border-radius:10px">
      <div class="funnel-num">{int(funnel_platform['cart'].sum()):,}</div>
      <div class="funnel-label">Add-to-Cart</div>
      <div class="funnel-rate">Add-to-Cart Rate (CTR) {avg_ctr:.1%}</div>
    </div>
    <div class="funnel-step" style="background:#fffbeb;border-radius:10px">
      <div class="funnel-num">{int(funnel_platform['order_behavior'].sum()):,}</div>
      <div class="funnel-label">Orders</div>
      <div class="funnel-rate">CVR {avg_cvr:.1%}</div>
    </div>
    <div class="funnel-step" style="background:#fdf4ff;border-radius:10px">
      <div class="funnel-num">{int(funnel_platform['payment'].sum()):,}</div>
      <div class="funnel-label">Payments</div>
      <div class="funnel-rate">Payment Rate {funnel_platform['payment_rate'].mean():.1%}</div>
    </div>
    <div class="funnel-step" style="background:#f0fdf4;border-radius:10px">
      <div class="funnel-num">¥{total_gmv:,.0f}</div>
      <div class="funnel-label">GMV</div>
    </div>
  </div>
</div>

<div class="section">
  <h2>GMV Ranking by Platform</h2>
  <table>
    <thead><tr><th>Platform</th><th>GMV (CNY)</th><th>Share</th></tr></thead>
    <tbody>{platform_rows}</tbody>
  </table>
</div>

<div class="section">
  <h2>Category GMV Top 10</h2>
  <table>
    <thead><tr><th>Category</th><th>GMV (CNY)</th><th>CTR</th><th>CVR</th><th>Share</th></tr></thead>
    <tbody>{category_rows}</tbody>
  </table>
</div>

<div class="section">
  <h2>Shapley Factor Attribution (by Platform)</h2>
  <p style="font-size:13px;color:#64748b;margin-bottom:14px">Positive = factor lifted GMV; Negative = factor dragged GMV. Green = positive, Red = negative.</p>
  <table>
    <thead><tr><th>Platform</th><th>Impression</th><th>CTR</th><th>CVR</th><th>ARPU</th></tr></thead>
    <tbody>{shapley_html if shapley_html else '<tr><td colspan=5 style="color:#94a3b8;text-align:center">Please run the Shapley computation cell first</td></tr>'}</tbody>
  </table>
</div>

<div class="section">
  <h2>Analysis Dimensions</h2>
  <span class="tag">Platform</span>
  <span class="tag">Category</span>
  <span class="tag">City</span>
  <span class="tag">Platform × Category</span>
  <span class="tag">Monthly Trend</span>
  <span class="tag">Funnel Conversion</span>
  <span class="tag">Shapley Attribution</span>
</div>

<div class="footer">E-commerce GMV Attribution System | For educational use only</div>

</body>
</html>"""
    return html


html_report = generate_html_report()
with open('gmv_report.html', 'w', encoding='utf-8') as f:
    f.write(html_report)
print('✅ HTML report generated: gmv_report.html')
display(HTML(html_report))



✅ HTML report generated: gmv_report.html


---
## 9. Summary & Output File List


In [19]:

print('=' * 55)
print('Output File List')
print('=' * 55)
print('  feishu_card.json    Feishu card JSON (paste directly into Feishu card builder)')
print('  gmv_report.html     HTML highlighted report (open in browser)')
print()
print('📌 Analysis Framework Recap:')
print('  GMV = Impression × CTR × CVR × ARPU')
print()
print('📌 Analysis Dimensions:')
print('  · Platform (taobao / jd / douyin)')
print('  · Product category (L1 + L2)')
print('  · City')
print('  · Platform × Category (2D cross)')
print()
print('📌 Algorithms:')
print('  · Multiplicative log-diff → MoM factor contributions')
print('  · Shapley Value → fair attribution weights')
print()
print('=' * 55)


Output File List
  feishu_card.json    Feishu card JSON (paste directly into Feishu card builder)
  gmv_report.html     HTML highlighted report (open in browser)

📌 Analysis Framework Recap:
  GMV = Impression × CTR × CVR × ARPU

📌 Analysis Dimensions:
  · Platform (taobao / jd / douyin)
  · Product category (L1 + L2)
  · City
  · Platform × Category (2D cross)

📌 Algorithms:
  · Multiplicative log-diff → MoM factor contributions
  · Shapley Value → fair attribution weights



---
## 10. Save Output Files


In [20]:
from pathlib import Path

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Export CSVs ─────────────────────────────────────────────────
funnel_platform.to_csv(OUTPUT_DIR / "platform_attribution.csv", index=False)
funnel_category.to_csv(OUTPUT_DIR / "category_attribution.csv", index=False)
funnel_city.to_csv(OUTPUT_DIR / "city_attribution.csv", index=False)
summary_2d.to_csv(OUTPUT_DIR / "platform_category_attribution.csv", index=False)
decomp_platform.to_csv(OUTPUT_DIR / "gmv_decomp_platform.csv", index=False)
decomp_category.to_csv(OUTPUT_DIR / "gmv_decomp_category.csv", index=False)
decomp_city.to_csv(OUTPUT_DIR / "gmv_decomp_city.csv", index=False)
shapley_df.reset_index().to_csv(OUTPUT_DIR / "shapley_attribution.csv", index=False)

# ── Export JSON ─────────────────────────────────────────────────
summary = {
    "total_gmv": round(order_valid["item_total"].sum(), 2),
    "top_platform": funnel_platform.groupby("platform")["gmv"].sum().idxmax(),
    "top_category": funnel_category.groupby("category")["gmv"].sum().idxmax(),
    "avg_ctr": round(funnel_platform["ctr"].mean(), 4),
    "avg_cvr": round(funnel_platform["cvr"].mean(), 4),
    "analysis_period": {
        "start": str(order_valid["order_time"].min()),
        "end": str(order_valid["order_time"].max()),
    },
}

with open(OUTPUT_DIR / "analysis_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

with open(OUTPUT_DIR / "feishu_card.json", "w", encoding="utf-8") as f:
    json.dump(feishu_card, f, ensure_ascii=False, indent=2, default=str)

print("Saved output files to:", OUTPUT_DIR.resolve())


Saved output files to: /Users/lauren/Downloads/output
